# Smart Claims Processor
## AI-Powered Insurance Claims Processing | LangGraph + CrewAI

This notebook walks through the full pipeline step by step.

**Frameworks:** LangGraph (orchestration) + CrewAI (fraud detection crew)  
**LLM:** Google Gemini 2.5 Flash (free tier)  
**Features:** Security (PII masking), Guardrails, HITL, LLM-as-Judge Evaluation


## Setup

In [ ]:
# Install dependencies
# !pip install -r ../requirements.txt

import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv
load_dotenv('../.env')

print('GOOGLE_API_KEY set:', bool(os.getenv('GOOGLE_API_KEY')))

## Part 1: Security - PII Masking

In [ ]:
from src.security.pii_masker import mask_claim, get_masked_summary

sample_claim = {
    'claim_id': 'CLM-2024-001234',
    'policy_number': 'POL-AUTO-789456',
    'claimant_name': 'Jane Smith',
    'claimant_email': 'jane.smith@gmail.com',
    'claimant_phone': '555-123-4567',
    'claimant_dob': '1985-03-15',
    'incident_description': 'Rear-ended at Main St. Contact jane.smith@gmail.com',
    'estimated_amount': 8500
}

masked = mask_claim(sample_claim)
print('Original name:', sample_claim['claimant_name'])
print('Masked name:', masked['claimant_name'])
print()
print('Original email:', sample_claim['claimant_email'])
print('Masked email:', masked['claimant_email'])
print()
print('Safe summary for LLM:')
print(get_masked_summary(sample_claim))

## Part 2: Fraud Pattern Detection (Rule-Based Fast Check)

In [ ]:
from src.tools.fraud_patterns import check_known_patterns, get_statistical_anomaly

# Clean claim
claim = {'incident_type': 'auto_collision', 'incident_date': '2024-11-15',
         'estimated_amount': 8500, 'police_report_number': 'APD-123'}
policy = {'start_date': '2023-01-01', 'claims_count': 0}

matched, score = check_known_patterns(claim, policy)
print(f'Clean claim - Fraud Score: {score:.2f}')
print(f'Matched patterns: {matched or "None"}')
print()

# Suspicious claim
suspicious = {'incident_type': 'auto_collision', 'incident_date': '2024-01-05',
              'estimated_amount': 15000, 'police_report_number': None}
new_policy = {'start_date': '2024-01-01', 'claims_count': 2}

matched2, score2 = check_known_patterns(suspicious, new_policy)
print(f'Suspicious claim - Fraud Score: {score2:.2f}')
print('Matched patterns:')
for p in matched2:
    print(f'  - {p}')

# Statistical anomaly check
print()
anomaly = get_statistical_anomaly('auto_collision', 50000)
print(f'$50K auto claim anomaly check: z-score={anomaly["z_score"]}, outlier={anomaly["is_outlier"]}')

## Part 3: HITL Trigger Logic

In [ ]:
from src.hitl.checkpoint import check_hitl_required
from src.models.schemas import FraudAssessmentOutput, FraudRiskLevel

claim = {'claim_id': 'CLM-001', 'policy_number': 'POL-001', 'estimated_amount': 25000, 'is_appeal': False}
fraud = FraudAssessmentOutput(
    fraud_risk_level=FraudRiskLevel.MEDIUM,
    fraud_score=0.35,
    primary_concerns=[],
    recommendation='proceed',
    crew_summary='Analysis complete',
    pattern_score=0.35, anomaly_score=0.35, consistency_score=0.70
)

req, triggers, priority, score = check_hitl_required(
    claim=claim, intake_output=None, fraud_output=fraud, damage_assessed_usd=22000
)

print(f'HITL Required: {req}')
print(f'Priority: {priority.value if req else "N/A"}')
print(f'Priority Score: {score}')
print('Triggers:')
for t in triggers:
    print(f'  - {t}')

## Part 4: Damage Calculator Tools

In [ ]:
from src.tools.damage_calculator import calculate_vehicle_acv, should_total_loss, apply_depreciation

# Calculate ACV for different vehicles
vehicles = [
    (2024, 'Toyota', 'Camry'),
    (2020, 'Honda', 'Civic'),
    (2015, 'Ford', 'Focus'),
    (2010, 'Chevy', 'Malibu'),
]

print('Vehicle Actual Cash Values:')
for year, make, model in vehicles:
    acv = calculate_vehicle_acv(year, make, model)
    print(f'  {year} {make} {model}: ${acv:,.2f}')

# Total loss check
print()
acv_2020 = calculate_vehicle_acv(2020, 'Honda', 'Civic')
repair_cost = 18000
is_tl, ratio = should_total_loss(repair_cost, acv_2020)
print(f'2020 Civic ACV: ${acv_2020:,.2f}')
print(f'Repair cost: ${repair_cost:,.2f}')
print(f'Total loss? {is_tl} (ratio: {ratio:.1%})')

# Depreciation
print()
depr_val, depr_amt = apply_depreciation(10000, 'auto', 5)
print(f'$10,000 damage on 5-year-old vehicle:')
print(f'  Depreciation applied: ${depr_amt:,.2f}')
print(f'  Net payable amount: ${depr_val:,.2f}')

## Part 5: Run Full Pipeline (requires GOOGLE_API_KEY)

In [ ]:
import json
from src.agents.graph import process_claim
from src.models.state import ClaimInput

# Load sample claim
with open('../data/sample_claims/auto_accident.json') as f:
    claim_data = json.load(f)

claim = ClaimInput(**claim_data)
print(f'Processing claim: {claim["claim_id"]}')
print(f'Policy: {claim["policy_number"]}')
print(f'Amount: ${claim["estimated_amount"]:,.2f}')
print(f'Type: {claim["incident_type"]}')

In [ ]:
# Process the claim (requires API key)
result = process_claim(claim)

decision = result.get('final_decision')
print('=' * 50)
print('PIPELINE RESULT')
print('=' * 50)
print(f'Decision: {decision}')
print(f'Settlement: ${result.get("final_amount_usd", 0):,.2f}')
print(f'HITL Required: {result.get("hitl_required")}')
print(f'Agent Calls: {result.get("agent_call_count")}')
print(f'Total Cost: ${result.get("total_cost_usd", 0):.4f}')

fraud = result.get('fraud_output')
if fraud:
    print(f'Fraud Score: {fraud.fraud_score:.2f} ({fraud.fraud_risk_level.value})')

eval_out = result.get('evaluation_output')
if eval_out:
    print(f'Evaluation Score: {eval_out.overall_score:.2f} ({"PASS" if eval_out.passed else "FAIL"})')

In [ ]:
# View the pipeline trace
print('Pipeline Trace:')
for step in result.get('pipeline_trace', []):
    print(f'  {step}')

In [ ]:
# View the claimant communication
comm = result.get('communication_output')
if comm:
    print('Subject:', comm.subject)
    print()
    print(comm.message)

## Part 6: Test All 4 Sample Scenarios

In [ ]:
import json
from pathlib import Path

scenarios = [
    ('auto_accident.json', 'Normal auto claim - expect APPROVED'),
    ('high_value_hitl.json', 'High value theft - expect HITL'),
    ('fraud_suspicious.json', 'Suspicious claim - expect high fraud score'),
    ('lapsed_policy.json', 'Lapsed policy - expect DENIED at intake'),
]

for filename, description in scenarios:
    path = Path(f'../data/sample_claims/{filename}')
    with open(path) as f:
        data = json.load(f)
    data.pop('_test_note', None)
    
    claim = ClaimInput(**data)
    result = process_claim(claim)
    
    decision = result.get('final_decision')
    decision_str = decision.value if hasattr(decision, 'value') else str(decision)
    fraud = result.get('fraud_output')
    
    print(f'[{filename}] {description}')
    print(f'  Decision: {decision_str} | Amount: ${result.get("final_amount_usd", 0):,.0f} | HITL: {result.get("hitl_required")} | Fraud: {fraud.fraud_score:.2f if fraud else "N/A"}')
    print()